# Dipole Angle vs Parallactic / Zenith Angle Correlation

This notebook investigates whether the **dipole position angle** (`r:dipoleAngle`) and the
**dipole separation** (`r:dipoleLength`) measured by the Fink/LSST difference-imaging pipeline
are correlated with observing-geometry angles:

| Observable | Definition |
|------------|------------|
| **Parallactic angle** η | Angle between North and the great circle toward the zenith, at the object position |
| **Zenith angle** z | Angular distance from zenith to object (complement of altitude) |

A correlation with the parallactic angle would hint at an **atmospheric dispersion** or
**differential refraction** origin for the dipoles; a correlation with the zenith angle
would point to an **airmass-dependent PSF** effect.

## Strategy

* Load dipole-only alerts from `data_DIPOLES_01c/` (same source as notebook `01d`).
* Compute η and z for each alert using `astropy` (RA, Dec, MJD → AltAz frame at Rubin).
* Study correlations **per DDF** and **per band**.
* Visualisations: scatter plots, 2-D histograms, polar plots, and Pearson/Spearman statistics.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-28

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u

warnings.filterwarnings("ignore")
print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Input data (written by notebook 01c) ─────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"

# ── Output figures ────────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_05"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── Rubin/LSST observatory location (Cerro Pachón) ───────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrees North
RUBIN_LON_DEG = -70.749417  # degrees East  (West is negative)
RUBIN_HEIGHT_M = 2647.0  # metres above sea level

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Observing-geometry helper functions

We compute, for each alert:
* **Parallactic angle** η — given directly by `AltAz.parallactic_angle` in astropy.
* **Zenith angle** z = 90° − altitude.
* **Airmass** X ≈ 1/cos(z) (for cross-checks).

In [ ]:
def compute_parallactic_zenith(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute the parallactic angle and zenith angle for a set of sky positions
    observed at given MJD times from a given ground location.

    Parameters
    ----------
    ra_deg, dec_deg : array-like
        ICRS coordinates in degrees.
    mjd : array-like
        Observation times in MJD (TAI).
    location : EarthLocation
        Observer position on Earth.
    batch_size : int
        Number of alerts processed per astropy call (trade-off speed vs memory).

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg : float  (−180 … +180 deg)
        zenith_angle_deg      : float  (0 … 90 deg)
        altitude_deg          : float
        azimuth_deg           : float
        airmass               : float  (≈ 1/cos(z))
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    za = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    az = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            times = Time(t[sl], format="mjd", scale="tai")
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)

            para[sl] = altaz.parallactic_angle().to(u.deg).value
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    # Simple flat-Earth airmass (valid for za < 80°)
    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "zenith_angle_deg": za,
            "altitude_deg": alt,
            "azimuth_deg": az,
            "airmass": airmass,
        }
    )


# Quick sanity check on a single alert
test = compute_parallactic_zenith(ra_deg=[150.1191], dec_deg=[2.2058], mjd=[60310.5])
print("Sanity check (COSMOS at MJD 60310.5):")
print(test.to_string(index=False))

## 3. Load dipole alerts from parquet files

We reuse the parquet files written by notebook `01c` and already used in `01d`.

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

NEEDED_COLS = [
    "r:ra",
    "r:dec",
    "r:midpointMjdTai",
    "r:isDipole",
    "r:dipoleAngle",
    "r:dipoleLength",
    "r:dipoleChi2",
    "r:dipoleFluxDiff",
    "r:band",
]

for field_name in DEEP_FIELDS:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue

    df = pd.read_parquet(pq)

    # Cast boolean column
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    # Cast numeric columns
    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep only dipole rows
    if "r:isDipole" in df.columns:
        df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    else:
        df_dip = pd.DataFrame()

    df_dip["field"] = field_name
    ddf_alerts[field_name] = df_dip

    print(f"[{field_name:12s}] {len(df):7,} total alerts  |  {len(df_dip):6,} dipoles")

print("\nLoad complete.")

## 4. Compute parallactic & zenith angles for every dipole alert

This step is the most compute-intensive: one `astropy` coordinate transformation per alert.
We process each DDF separately and cache results in a new column.

In [ ]:
frames_with_angles: list[pd.DataFrame] = []

for field_name, df_dip in ddf_alerts.items():
    if df_dip.empty:
        print(f"[{field_name:12s}] no dipoles — skipping.")
        continue

    # Check required columns
    missing = [c for c in ("r:ra", "r:dec", "r:midpointMjdTai") if c not in df_dip.columns]
    if missing:
        print(f"[{field_name:12s}] missing columns {missing} — skipping.")
        continue

    # Drop rows with NaN coordinates or time
    mask = df_dip["r:ra"].notna() & df_dip["r:dec"].notna() & df_dip["r:midpointMjdTai"].notna()
    df_clean = df_dip[mask].copy().reset_index(drop=True)

    print(f"[{field_name:12s}] computing angles for {len(df_clean):,} dipoles ...", end=" ")

    geo = compute_parallactic_zenith(
        ra_deg=df_clean["r:ra"].values,
        dec_deg=df_clean["r:dec"].values,
        mjd=df_clean["r:midpointMjdTai"].values,
    )

    df_clean = pd.concat([df_clean.reset_index(drop=True), geo.reset_index(drop=True)], axis=1)
    frames_with_angles.append(df_clean)
    print("done")

# Concatenated catalogue of all dipoles with geometry
if frames_with_angles:
    df_all = pd.concat(frames_with_angles, ignore_index=True)
    print(f"\nTotal dipoles with angles: {len(df_all):,}")
    print(
        df_all[
            ["field", "r:band", "parallactic_angle_deg", "zenith_angle_deg", "altitude_deg", "airmass"]
        ].describe()
    )
else:
    df_all = pd.DataFrame()
    print("No dipoles found — nothing to analyse.")

## 5. Overview distributions

Quick look at the distribution of parallactic and zenith angles across all DDFs.

In [ ]:
if df_all.empty:
    print("No data — skipping.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))

    axes[0].hist(
        df_all["parallactic_angle_deg"].dropna(), bins=36, color="steelblue", edgecolor="white", lw=0.3
    )
    axes[0].set_xlabel("Parallactic angle η (deg)")
    axes[0].set_ylabel("N dipoles")
    axes[0].set_title("Parallactic angle distribution")

    axes[1].hist(df_all["zenith_angle_deg"].dropna(), bins=30, color="darkorange", edgecolor="white", lw=0.3)
    axes[1].set_xlabel("Zenith angle z (deg)")
    axes[1].set_title("Zenith angle distribution")

    axes[2].hist(df_all["airmass"].dropna().clip(1, 3), bins=30, color="seagreen", edgecolor="white", lw=0.3)
    axes[2].set_xlabel("Airmass X")
    axes[2].set_title("Airmass distribution")

    axes[3].hist(df_all["r:dipoleAngle"].dropna() % 360, bins=36, color="crimson", edgecolor="white", lw=0.3)
    axes[3].set_xlabel("Dipole angle (deg)")
    axes[3].set_title("Dipole angle distribution")

    plt.suptitle("All DDFs combined — overview", y=1.02, fontsize=11)
    plt.tight_layout()
    savefig("overview_distributions")
    plt.show()

## 6. Dipole angle vs parallactic angle — scatter & 2D histogram

**Per DDF**, coloured by band.

In [ ]:
def plot_dipole_vs_angle(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    field_name: str,
    figname_suffix: str,
) -> None:
    """
    Two-panel figure:
      Left  — scatter (x_col vs dipoleAngle) coloured by band
      Right — 2D histogram density
    Also prints Pearson and Spearman correlation coefficients.
    """
    # Drop NaNs
    mask = df[x_col].notna() & df["r:dipoleAngle"].notna()
    sub = df[mask].copy()
    if len(sub) < 5:
        print(f"  [{field_name}] too few points — skipping {x_col}.")
        return

    x = sub[x_col].values
    y = sub["r:dipoleAngle"].values % 360.0

    # Correlation statistics
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    stat_str = f"Pearson r={r_p:.3f} (p={p_p:.2e})   Spearman ρ={r_s:.3f} (p={p_s:.2e})"
    print(f"  [{field_name}] {x_label}:  {stat_str}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # ── Scatter coloured by band ───────────────────────────────────────────────
    if "r:band" in sub.columns:
        bands_present = [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]
        for band in bands_present:
            idx = sub["r:band"] == band
            ax1.scatter(
                sub.loc[idx, x_col],
                sub.loc[idx, "r:dipoleAngle"] % 360.0,
                s=4,
                alpha=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=f"{band} (n={idx.sum():,})",
                rasterized=True,
            )
        ax1.legend(fontsize=7, markerscale=2, loc="best")
    else:
        ax1.scatter(x, y, s=4, alpha=0.4, color="steelblue", rasterized=True)

    ax1.set_xlabel(x_label)
    ax1.set_ylabel("Dipole angle (deg)")
    ax1.set_title(f"{field_name} — scatter\n{stat_str}", fontsize=8)

    # ── 2D histogram ─────────────────────────────────────────────────────────
    h2 = ax2.hist2d(
        x,
        y,
        bins=[40, 36],
        cmap="viridis",
        norm=mcolors.LogNorm(vmin=1),
    )
    plt.colorbar(h2[3], ax=ax2, label="N dipoles (log scale)")
    ax2.set_xlabel(x_label)
    ax2.set_ylabel("Dipole angle (deg)")
    ax2.set_title(f"{field_name} — 2D histogram")

    plt.tight_layout()
    safe_field = field_name.replace("-", "_")
    savefig(f"{safe_field}_{figname_suffix}")
    plt.show()


print("Helper function defined.")

In [ ]:
# Dipole angle vs parallactic angle — per DDF
print("=" * 70)
print("Dipole angle vs PARALLACTIC angle — per DDF")
print("=" * 70)

for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        print(f"  [{field_name}] no data — skipping.")
        continue
    plot_dipole_vs_angle(
        df=sub,
        x_col="parallactic_angle_deg",
        x_label="Parallactic angle η (deg)",
        field_name=field_name,
        figname_suffix="dipoleAngle_vs_parallactic",
    )

In [ ]:
# Dipole angle vs zenith angle — per DDF
print("=" * 70)
print("Dipole angle vs ZENITH angle — per DDF")
print("=" * 70)

for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        print(f"  [{field_name}] no data — skipping.")
        continue
    plot_dipole_vs_angle(
        df=sub,
        x_col="zenith_angle_deg",
        x_label="Zenith angle z (deg)",
        field_name=field_name,
        figname_suffix="dipoleAngle_vs_zenith",
    )

## 7. Dipole length vs parallactic & zenith angles

If dipoles are caused by differential refraction, their **length** should grow with airmass /
zenith angle, independently of their direction.

In [ ]:
def plot_length_vs_angle(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    field_name: str,
    figname_suffix: str,
) -> None:
    """Scatter and 2D histogram of dipole length vs an observing-geometry angle."""
    mask = df[x_col].notna() & df["r:dipoleLength"].notna()
    sub = df[mask].copy()
    if len(sub) < 5:
        return

    x = sub[x_col].values
    y = sub["r:dipoleLength"].values

    # Clip extreme outliers for display
    y_clip = np.clip(y, 0, np.nanpercentile(y, 99))

    r_s, p_s = stats.spearmanr(x, y)
    print(f"  [{field_name}] {x_label} vs length:  Spearman ρ={r_s:.3f} (p={p_s:.2e})")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    if "r:band" in sub.columns:
        bands_present = [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]
        for band in bands_present:
            idx = sub["r:band"] == band
            ax1.scatter(
                sub.loc[idx, x_col],
                np.clip(sub.loc[idx, "r:dipoleLength"].values, 0, np.nanpercentile(y, 99)),
                s=4,
                alpha=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                rasterized=True,
            )
        ax1.legend(fontsize=7, markerscale=2)
    else:
        ax1.scatter(x, y_clip, s=4, alpha=0.4, color="steelblue", rasterized=True)

    ax1.set_xlabel(x_label)
    ax1.set_ylabel("Dipole length (arcsec)")
    ax1.set_title(f"{field_name} — length vs {x_label}\nSpearman ρ={r_s:.3f} (p={p_s:.2e})", fontsize=8)

    h2 = ax2.hist2d(x, y_clip, bins=[40, 30], cmap="plasma", norm=mcolors.LogNorm(vmin=1))
    plt.colorbar(h2[3], ax=ax2, label="N dipoles (log scale)")
    ax2.set_xlabel(x_label)
    ax2.set_ylabel("Dipole length (arcsec)")
    ax2.set_title(f"{field_name} — 2D histogram")

    plt.tight_layout()
    safe_field = field_name.replace("-", "_")
    savefig(f"{safe_field}_{figname_suffix}")
    plt.show()


for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        continue
    plot_length_vs_angle(
        sub, "parallactic_angle_deg", "Parallactic angle η (deg)", field_name, "dipoleLength_vs_parallactic"
    )
    plot_length_vs_angle(
        sub, "zenith_angle_deg", "Zenith angle z (deg)", field_name, "dipoleLength_vs_zenith"
    )
    plot_length_vs_angle(sub, "airmass", "Airmass X", field_name, "dipoleLength_vs_airmass")

## 8. Angular difference: dipole angle − parallactic angle

If atmospheric dispersion drives the dipoles, the **relative angle**
Δ = (dipoleAngle − η) mod 360° should cluster near 0° or 180°
(dipole axis aligned / anti-aligned with the direction to zenith).

We compute Δ for each alert and plot its polar distribution per DDF and per band.

In [ ]:
if not df_all.empty:
    # Compute delta angle (wrapped to 0–360 and folded to 0–180 for a headless vector)
    df_all["delta_angle_deg"] = (df_all["r:dipoleAngle"] - df_all["parallactic_angle_deg"]) % 360.0
    df_all["delta_angle_folded_deg"] = df_all["delta_angle_deg"].apply(
        lambda a: a if a <= 180.0 else 360.0 - a
    )
    print("delta_angle_deg column created.")
    print(df_all["delta_angle_deg"].describe())

In [ ]:
def polar_rose(df: pd.DataFrame, angle_col: str, title: str, figname: str, n_bins: int = 36) -> None:
    """Polar histogram of *angle_col* (in degrees), stacked by band."""
    if df.empty or angle_col not in df.columns:
        return

    bin_edges = np.linspace(0, 360, n_bins + 1)
    bin_edges_rad = np.radians(bin_edges)
    centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width_rad = 2 * np.pi / n_bins

    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="polar")

    bottom = np.zeros(n_bins)
    n_total = 0

    if "r:band" in df.columns:
        bands_present = [b for b in BAND_ORDER if b in df["r:band"].dropna().unique()]
        for band in bands_present:
            vals = df.loc[df["r:band"] == band, angle_col].dropna().values % 360.0
            if len(vals) == 0:
                continue
            n_total += len(vals)
            cnts, _ = np.histogram(vals, bins=bin_edges)
            ax.bar(
                centers_rad,
                cnts,
                width=width_rad * 0.9,
                bottom=bottom,
                color=BAND_COLORS.get(band, "grey"),
                edgecolor="white",
                linewidth=0.4,
                alpha=0.85,
                label=f"{band} (n={len(vals):,})",
            )
            bottom += cnts
        ax.legend(loc="lower right", fontsize=7, bbox_to_anchor=(1.30, -0.05))
    else:
        vals = df[angle_col].dropna().values % 360.0
        n_total = len(vals)
        cnts, _ = np.histogram(vals, bins=bin_edges)
        ax.bar(centers_rad, cnts, width=width_rad * 0.9, color="steelblue", edgecolor="white", linewidth=0.4)

    if n_total > 0:
        uniform = n_total / n_bins
        ax.plot(
            np.linspace(0, 2 * np.pi, 300),
            np.full(300, uniform),
            "--",
            color="crimson",
            lw=1.2,
            alpha=0.85,
            label="uniform",
        )

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(f"{title}\n(n={n_total:,})", va="bottom", pad=20)
    plt.tight_layout()
    savefig(figname)
    plt.show()


# Global polar rose of Δ
polar_rose(df_all, "delta_angle_deg", "Dipole angle − Parallactic angle (all DDFs)", "all_delta_angle_polar")

# Per-DDF polar roses
for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        continue
    polar_rose(
        sub,
        "delta_angle_deg",
        f"{field_name} — Δ(dipoleAngle − η)",
        f"{field_name.replace('-', '_')}_delta_angle_polar",
    )

## 9. Band-resolved correlation summary

For each DDF × band combination, compute the Pearson and Spearman correlations between
`dipoleAngle` and {`parallactic_angle`, `zenith_angle`}, and display the result as a heatmap.

In [ ]:
rows = []

for field_name in DEEP_FIELDS:
    sub_field = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub_field.empty:
        continue

    bands_present = sub_field["r:band"].dropna().unique() if "r:band" in sub_field.columns else ["all"]
    for band in bands_present:
        sub = sub_field[sub_field["r:band"] == band] if band != "all" else sub_field

        for x_col, x_label in [
            ("parallactic_angle_deg", "parallactic"),
            ("zenith_angle_deg", "zenith"),
        ]:
            for y_col, y_label in [
                ("r:dipoleAngle", "dipoleAngle"),
                ("r:dipoleLength", "dipoleLength"),
            ]:
                mask = sub[x_col].notna() & sub[y_col].notna()
                x = sub.loc[mask, x_col].values
                y = sub.loc[mask, y_col].values
                if len(x) < 5:
                    continue
                r_p, p_p = stats.pearsonr(x, y)
                r_s, p_s = stats.spearmanr(x, y)
                rows.append(
                    {
                        "field": field_name,
                        "band": band,
                        "x": x_label,
                        "y": y_label,
                        "n": int(mask.sum()),
                        "pearson_r": round(r_p, 4),
                        "pearson_p": round(p_p, 4),
                        "spearman_r": round(r_s, 4),
                        "spearman_p": round(p_s, 4),
                    }
                )

df_corr = pd.DataFrame(rows)
if not df_corr.empty:
    print("Correlation summary (all field × band × variable combinations):")
    pd.set_option("display.max_rows", 100)
    display(df_corr.sort_values(["field", "band", "x", "y"]))
else:
    print("No correlation data computed.")

In [ ]:
# Heatmap of Spearman ρ(dipoleAngle, parallactic_angle) per DDF × band
if not df_corr.empty:
    for y_label, x_label, title_suffix in [
        ("dipoleAngle", "parallactic", "DipoleAngle vs Parallactic"),
        ("dipoleAngle", "zenith", "DipoleAngle vs Zenith"),
        ("dipoleLength", "parallactic", "DipoleLength vs Parallactic"),
        ("dipoleLength", "zenith", "DipoleLength vs Zenith"),
    ]:
        sub_heat = df_corr[(df_corr["x"] == x_label) & (df_corr["y"] == y_label)]
        if sub_heat.empty:
            continue

        pivot = sub_heat.pivot_table(index="field", columns="band", values="spearman_r").reindex(
            columns=BAND_ORDER
        )

        fig, ax = plt.subplots(figsize=(8, max(3, len(pivot) * 0.6)))
        im = ax.imshow(pivot.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, label="Spearman ρ")
        ax.set_xticks(range(pivot.shape[1]))
        ax.set_xticklabels(pivot.columns.tolist())
        ax.set_yticks(range(pivot.shape[0]))
        ax.set_yticklabels(pivot.index.tolist())
        ax.set_xlabel("Band")
        ax.set_ylabel("DDF")
        ax.set_title(f"Spearman ρ — {title_suffix}")

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color="black")

        plt.tight_layout()
        tag = title_suffix.lower().replace(" ", "_")
        savefig(f"heatmap_spearman_{tag}")
        plt.show()

## 10. Dipole angle vs parallactic angle — overlaid per band (all DDFs)

One figure per band showing the scatter for all DDFs combined, to check whether
any wavelength-dependent trend is visible.

In [ ]:
if df_all.empty or "r:band" not in df_all.columns:
    print("No data — skipping per-band plots.")
else:
    bands_present = [b for b in BAND_ORDER if b in df_all["r:band"].dropna().unique()]

    for x_col, x_label, figname_base in [
        ("parallactic_angle_deg", "Parallactic angle η (deg)", "all_ddfs_dipoleAngle_vs_parallactic_by_band"),
        ("zenith_angle_deg", "Zenith angle z (deg)", "all_ddfs_dipoleAngle_vs_zenith_by_band"),
    ]:
        ncols = min(3, len(bands_present))
        nrows = int(np.ceil(len(bands_present) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), squeeze=False)

        for idx, band in enumerate(bands_present):
            ax = axes[idx // ncols][idx % ncols]
            sub = df_all[(df_all["r:band"] == band) & df_all[x_col].notna() & df_all["r:dipoleAngle"].notna()]

            if len(sub) < 5:
                ax.set_visible(False)
                continue

            x = sub[x_col].values
            y = sub["r:dipoleAngle"].values % 360.0
            r_s, p_s = stats.spearmanr(x, y)

            # Colour points by DDF
            field_list = sorted(sub["field"].dropna().unique())
            cmap_f = plt.get_cmap("tab10", len(field_list))
            for k, fld in enumerate(field_list):
                idx_f = sub["field"] == fld
                ax.scatter(
                    sub.loc[idx_f, x_col],
                    sub.loc[idx_f, "r:dipoleAngle"] % 360.0,
                    s=4,
                    alpha=0.4,
                    color=cmap_f(k),
                    label=fld,
                    rasterized=True,
                )

            ax.set_xlabel(x_label, fontsize=8)
            ax.set_ylabel("Dipole angle (deg)", fontsize=8)
            ax.set_title(
                f"Band {band}  (n={len(sub):,})\nSpearman ρ={r_s:.3f}  p={p_s:.2e}",
                fontsize=8,
            )
            ax.legend(fontsize=6, markerscale=2, loc="best")

        # Hide unused axes
        for idx in range(len(bands_present), nrows * ncols):
            axes[idx // ncols][idx % ncols].set_visible(False)

        plt.suptitle(f"Dipole angle vs {x_label} — all DDFs, per band", y=1.01, fontsize=11)
        plt.tight_layout()
        savefig(figname_base)
        plt.show()

## 11. Summary

| Section | What is shown |
|---------|---------------|
| §5 | Overview distributions of η, z, airmass, dipole angle |
| §6 | Scatter + 2D histogram: dipole angle vs η and vs z (per DDF) |
| §7 | Scatter + 2D histogram: dipole length vs η, z, airmass (per DDF) |
| §8 | Polar rose of Δ = dipoleAngle − η (globally and per DDF) |
| §9 | Pearson/Spearman ρ table and heatmap for all DDF × band combinations |
| §10 | Per-band panels: dipole angle vs η and vs z (all DDFs combined) |

A significant concentration of Δ near 0° or 180°, or a positive Spearman ρ between
dipole length and airmass, would be strong evidence for an **atmospheric origin** of the dipoles.
